In [3]:
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# 1. Generate Synthetic Enterprise Leads & Context Knowledge Base
synthetic_leads = pd.DataFrame([
    {
        "lead_id": "L-8392",
        "company": "NexusCorp Logistics",
        "user_query": "Hi, we want to scale our data pipeline processing speed. Does your platform support real-time stream encryption, and how is it priced?",
        "intent_priority": "High"
    },
    {
        "lead_id": "L-4711",
        "company": "Apex Financial",
        "user_query": "Hello, we need a solution for automated risk reporting. Do you integrate with legacy banking structures?",
        "intent_priority": "Medium"
    }
])

synthetic_kb = {
    "encryption": "Enterprise platform supports AES-256 real-time stream encryption with custom KMS key rotations. Pricing scales linearly at $0.02 per gigabyte processed.",
    "integrations": "We support legacy financial messaging layers including SWIFT MT/MX and ISO 20022 schemas via our standard secure banking gateway adapters."
}

# 2. Initialize Hugging Face Models and Pipelines
print("Initializing Hugging Face Text Generation Pipelines...")
# Utilizing a compact, instruction-tuned architecture suited for sandbox execution
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# 3. Process Leads Through the Agentic Pipeline
def run_agent_outreach_pipeline(lead_row):
    query = lead_row['user_query']
    company = lead_row['company']

    # Simple semantic router to look up relevant knowledge base context
    kb_context = ""
    if "encrypt" in query.lower() or "price" in query.lower():
        kb_context = synthetic_kb["encryption"]
    elif "legacy" in query.lower() or "bank" in query.lower():
        kb_context = synthetic_kb["integrations"]
    else:
        kb_context = "Standard corporate overview documentation."

    # Structured Prompt Engineering using explicit role guidelines
    messages = [
        {"role": "system", "content": "You are an expert enterprise sales agent. You write grounded, accurate email drafts using only the provided facts. Never hallucinate or promise unverified features."},
        {"role": "user", "content": f"Company: {company}\nInbound Query: {query}\nVerified Product Facts: {kb_context}\n\nTask: Write a concise, professional follow-up email response addressing the prospect's explicit questions based on our product facts."}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = generator(prompt, max_new_tokens=256, do_sample=True, temperature=0.2, top_p=0.9)
    email_draft = outputs[0]['generated_text'][len(prompt):]

    return email_draft, kb_context

# Execute across synthetic dataframe
for idx, row in synthetic_leads.iterrows():
    print(f"\n=== PROCESSING LEAD {row['lead_id']} FOR {row['company']} ===")
    draft, context = run_agent_outreach_pipeline(row)
    print(f"Retrieved KB Grounding Context: {context}")



Initializing Hugging Face Text Generation Pipelines...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== PROCESSING LEAD L-8392 FOR NexusCorp Logistics ===


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved KB Grounding Context: Enterprise platform supports AES-256 real-time stream encryption with custom KMS key rotations. Pricing scales linearly at $0.02 per gigabyte processed.

=== PROCESSING LEAD L-4711 FOR Apex Financial ===
Retrieved KB Grounding Context: We support legacy financial messaging layers including SWIFT MT/MX and ISO 20022 schemas via our standard secure banking gateway adapters.


In [4]:
    print(f"Generated Grounded Outreach Email Draft:\n{draft.strip()}")

Generated Grounded Outreach Email Draft:
Subject: Automated Risk Reporting Solution Integration

Dear [Prospect's Name],

Thank you for your inquiry regarding our automated risk reporting solution. I hope this message finds you well.

We understand that integrating with existing legacy banking structures is crucial for many organizations. Our platform supports various financial messaging layers, including SWIFT MT/MX and ISO 20022 schemas through our standard secure banking gateway adapters. This integration allows us to efficiently process and report data from these systems, ensuring seamless compatibility with your current infrastructure.

Would you be interested in exploring how our solution can meet your specific needs? Please let me know if there’s any additional information you require about our capabilities or pricing options.

Looking forward to potentially partnering with you to enhance your risk management processes.

Best regards,

[Your Full Name]  
[Your Position]  
Apex F